# TensorFlow Parity: Keras Mass Regression

In [1]:
from pathlib import Path

import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path.cwd() / "data"
eu_df = pd.read_csv(
    DATA_DIR / "exoplanet.eu_catalog_28-05-26_22_04_09.csv", low_memory=False
)
tf.__version__

2026-06-03 17:37:57.913727: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-03 17:37:58.673179: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-03 17:37:59.010877: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-03 17:37:59.012583: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-03 17:37:59.482993: I tensorflow/core/platform/cpu_feature_gua

'2.16.2'

In [2]:
target = "mass"
feature_columns = [
    "mass_sini",
    "radius",
    "orbital_period",
    "semi_major_axis",
    "eccentricity",
    "inclination",
    "temp_calculated",
    "star_distance",
    "star_metallicity",
    "star_mass",
    "star_radius",
    "star_age",
    "star_teff",
    "planet_status",
    "detection_type",
    "mass_measurement_type",
    "star_sp_type",
]

model_df = eu_df[[target, *feature_columns]].copy()
model_df = model_df[model_df[target].notna()]

X = pd.get_dummies(model_df.drop(columns=[target]), dummy_na=True)
X = X.fillna(X.median(numeric_only=True)).fillna(0.0)
y = model_df[target].astype("float32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [3]:
keras_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1),
    ]
)

keras_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
keras_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        38,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,593 (182.00 KB)

 Trainable params: 46,593 (182.00 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
history = keras_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    verbose=0,
)

pd.DataFrame(history.history).tail()

,loss,mae,val_loss,val_mae
25,2.374898,0.863549,5.272967,1.182261
26,2.309225,0.856037,5.225078,1.208478
27,2.236315,0.837717,5.217187,1.213425
28,2.186862,0.827750,5.300370,1.204296
29,2.232386,0.819511,5.275225,1.238233


In [5]:
evaluation = keras_model.evaluate(X_test_scaled, y_test, verbose=0)
{"loss": evaluation[0], "mae": evaluation[1]}

{'loss': 3.6081361770629883, 'mae': 1.133626103401184}